# UPAR 2027 Track 2 — CSAR pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/UPAR-Challenge-2027/blob/main/notebooks/pipeline.ipynb)

Data → score study → backbone → training → evaluation and calibration → export → submission test, with the same entry points as the CLI (`python -m src.train`, ...).

* `MODE = "smoke"`: synthetic data (real labels, random images), CPU, about 1 minute.
* `MODE = "full"`: real images, downloaded if missing. Use a GPU (Colab: *Runtime → Change runtime type → GPU*).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/UPAR-Challenge-2027.git"
IN_COLAB = "google.colab" in sys.modules

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src" / "train.py").is_file():
        ROOT = candidate
        break
else:
    ROOT = Path("/content/UPAR-Challenge-2027") if IN_COLAB else Path.cwd() / "UPAR-Challenge-2027"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
import time

import numpy as np
import torch

MODE = "smoke"  # "smoke" | "full"
BACKBONE = "resnet18" if MODE == "smoke" else "convnext_base"
HOLDOUT = None  # None (final model) | "Market1501" | "PA100k" | "PETA" (LODO fold)
EPOCHS = 1 if MODE == "smoke" else 20
BATCH_SIZE = 16 if MODE == "smoke" else 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 0 if MODE == "smoke" else 8
RUN_DIR = ROOT / "runs" / f"{MODE}_{BACKBONE}{'_' + HOLDOUT if HOLDOUT else ''}"
CHECKPOINT = RUN_DIR / "model.pt"
print(f"mode={MODE} backbone={BACKBONE} holdout={HOLDOUT} device={DEVICE} run={RUN_DIR}")

## 1. Data

In [ ]:
from src.data import load_split
from src.synthetic import make_synthetic_data

if MODE == "smoke":
    DATA_DIR = make_synthetic_data(ROOT / "data", RUN_DIR / "data", n_train=96, n_val=48)
else:
    DATA_DIR = ROOT / "data"
    if not (DATA_DIR / "Market1501").is_dir():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tqdm"], check=True)
        subprocess.run([sys.executable, "download_datasets.py", "--data-dir", str(DATA_DIR)], check=True)

for name in ("train", "val"):
    split = load_split(DATA_DIR, name)
    domains, counts = np.unique(split.domains, return_counts=True)
    missing = sum(not (DATA_DIR / p).is_file() for p in split.images[:200])
    print(f"{name}: {len(split)} images, {len(split.queries)} queries, "
          f"{dict(zip(domains.tolist(), counts.tolist()))}, missing images (first 200): {missing}")

## 2. Score functions

Official validation ground truth plus Gaussian logit noise, no images needed: `l1` (official example), `loglik`, `structured`, `mix` (E[ndom] + λ·P_exact).

In [ ]:
from src import evaluate

evaluate.main(["--data-dir", str(ROOT / "data"), "--simulate", "3.0",
               "--scores", "l1", "loglik", "structured", "mix", "--lams", "1", "30",
               "--max-queries", "300", "--no-domains"])

## 3. Backbone

Loads the ImageNet weights and estimates the inference time of a 28,000-image gallery (2024 test size) on the current hardware.

In [ ]:
from src import benchmark

benchmark.main(["--backbones", BACKBONE, "--device", DEVICE, "--batch-size", "8", "--iters", "1"])

## 4. Training

In [ ]:
from src import train

train_args = ["--data-dir", str(DATA_DIR), "--out", str(RUN_DIR), "--backbone", BACKBONE,
              "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
              "--workers", str(WORKERS), "--device", DEVICE]
if HOLDOUT:
    train_args += ["--holdout", HOLDOUT]
if MODE == "smoke":
    train_args += ["--no-pretrained", "--height", "128", "--width", "64"]
train.main(train_args)

## 5. Evaluation, calibration and score selection

On a LODO fold, calibration is refitted on half of the held-out queries and measured on the other half. `--save-best` writes the best score and calibration into the checkpoint.

In [ ]:
eval_args = ["--data-dir", str(DATA_DIR), "--checkpoint", str(CHECKPOINT),
             "--scores", "loglik", "structured", "mix", "--lams", "1", "30", "--gammas", "0", "0.5",
             "--device", DEVICE, "--workers", str(WORKERS), "--save-best"]
if HOLDOUT:
    eval_args += ["--holdout", HOLDOUT, "--calib-frac", "0.5"]
if MODE == "smoke":
    eval_args += ["--no-domains"]
evaluate.main(eval_args)

## 6. Export

In [ ]:
from src import export

archive = export.export(CHECKPOINT, RUN_DIR / "submission", overwrite=True)
print(archive, f"{archive.stat().st_size / 2**20:.1f} MiB")

## 7. Submission test

Imports the exported `run.py` (not the repository code), calls `rank_gallery` like the ingestion program, then computes the official metrics.

In [ ]:
import importlib.util

from src.attributes import ATTRIBUTE_NAMES
from src.data import query_mask
from src.metrics import evaluate_output

spec = importlib.util.spec_from_file_location("submission_run", RUN_DIR / "submission" / "run.py")
submission = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission)
submission.WORKERS = WORKERS

gallery = load_split(DATA_DIR, "val")
if HOLDOUT:
    gallery = gallery.subset(gallery.domains == HOLDOUT)
if len(gallery.queries) > 367:
    gallery = gallery.subset(query_mask(gallery, num_queries=367))
sample = {"attribute_names": list(ATTRIBUTE_NAMES),
          "queries": gallery.queries.tolist(),
          "gallery": [{"image_path": str(DATA_DIR / p)} for p in gallery.images]}

t0 = time.time()
output = submission.rank_gallery(sample)
print(f"{len(gallery.queries)} queries x {len(gallery)} images in {time.time() - t0:.1f}s")
evaluate_output(output, gallery.queries, gallery.labels)

## Next steps

1. Run the three LODO folds (`HOLDOUT = "Market1501"`, `"PA100k"`, `"PETA"`) and compare scores fold by fold.
2. Train the final model with `HOLDOUT = None` and submit `runs/full_<backbone>/submission.zip`.
3. Keep `--transductive` off until the organisers approve it in writing.